In [1]:
#Install dependenecies
# %pip install anthropic python-dotenv

In [2]:
#Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
#Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

In [4]:
def add_user_message(messages, text):
    user_message = { "role": "user", "content": text }
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = { "role": "assistant", "content": text }
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    message = client.messages.create(**params)
    return message.content[0].text

In [5]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [6]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

C:\Users\gabri\AppData\Local\Temp\ipykernel_6636\1556694884.py:23: DeprecationWarning: The model 'claude-sonnet-4-0' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  message = client.messages.create(**params)


In [7]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
                Please solve the following task: 

                {test_case['task']}
            """
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [8]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [9]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [10]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

C:\Users\gabri\AppData\Local\Temp\ipykernel_6636\1556694884.py:23: DeprecationWarning: The model 'claude-sonnet-4-0' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  message = client.messages.create(**params)


In [11]:
print(json.dumps(results, indent=2))

[
  {
    "output": "Here's a Python function that extracts the bucket name from an S3 bucket ARN:\n\n```python\ndef extract_bucket_name_from_arn(arn):\n    \"\"\"\n    Extract the bucket name from an S3 bucket ARN.\n    \n    Args:\n        arn (str): The S3 bucket ARN in format 'arn:aws:s3:::bucket-name'\n    \n    Returns:\n        str: The bucket name\n    \n    Raises:\n        ValueError: If the ARN format is invalid\n    \"\"\"\n    if not isinstance(arn, str):\n        raise ValueError(\"ARN must be a string\")\n    \n    # Split the ARN by colons\n    arn_parts = arn.split(':')\n    \n    # Validate ARN format\n    if len(arn_parts) != 6:\n        raise ValueError(\"Invalid ARN format. Expected format: 'arn:aws:s3:::bucket-name'\")\n    \n    if arn_parts[0] != 'arn':\n        raise ValueError(\"ARN must start with 'arn'\")\n    \n    if arn_parts[1] != 'aws':\n        raise ValueError(\"ARN must contain 'aws' as partition\")\n    \n    if arn_parts[2] != 's3':\n        raise 